In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Reference date — all features are computed relative to this
# This is your observation cutoff (no data beyond this point)
CUTOFF = pd.Timestamp('2023-09-30')

# ── Load all tables ────────────────────────────────────────
customers    = pd.read_csv(r'/Users/siddharthm/Desktop/churn-prediction/data/processed/telco_cleaned.csv')
transactions = pd.read_csv(r'/Users/siddharthm/Desktop/churn-prediction/data/synthetic/transactions.csv',
                           parse_dates=['transaction_date'])
sessions     = pd.read_csv(r'/Users/siddharthm/Desktop/churn-prediction/data/synthetic/sessions.csv',
                           parse_dates=['session_start'])
tickets      = pd.read_csv(r'/Users/siddharthm/Desktop/churn-prediction/data/synthetic/support_tickets.csv',
                           parse_dates=['created_date', 'resolved_date'])

print("✅ Tables loaded")
print(f"   customers:    {len(customers):,} rows")
print(f"   transactions: {len(transactions):,} rows")
print(f"   sessions:     {len(sessions):,} rows")
print(f"   tickets:      {len(tickets):,} rows")
print(f"\n   Observation cutoff: {CUTOFF.date()}")
print(f"   All features will be computed up to this date only.")

✅ Tables loaded
   customers:    7,043 rows
   transactions: 202,377 rows
   sessions:     1,218,789 rows
   tickets:      41,190 rows

   Observation cutoff: 2023-09-30
   All features will be computed up to this date only.


In [2]:
# ── Cell 2: RFM Features from Transactions ────────────────

# Work only with data up to the cutoff (already enforced in generator,
# but filtering here makes the intent explicit)
txn = transactions[transactions['transaction_date'] <= CUTOFF].copy()

# ── Recency: days since last transaction ──────────────────
last_txn = txn.groupby('customer_id')['transaction_date'].max().reset_index()
last_txn.columns = ['customer_id', 'last_transaction_date']
last_txn['recency_days'] = (CUTOFF - last_txn['last_transaction_date']).dt.days

# ── Frequency: total number of transactions ───────────────
freq = txn.groupby('customer_id').size().reset_index(name='transaction_count')

# ── Monetary: total and average spend ─────────────────────
monetary = txn.groupby('customer_id').agg(
    total_spend   = ('amount', 'sum'),
    avg_spend     = ('amount', 'mean'),
    spend_std     = ('amount', 'std')   # spend variability (erratic payments = risk)
).reset_index()
monetary['spend_std'] = monetary['spend_std'].fillna(0)

# ── Payment health features ───────────────────────────────
payment = txn.groupby('customer_id').agg(
    failed_payments     = ('payment_status', lambda x: (x == 'failed').sum()),
    total_payments      = ('payment_status', 'count')
).reset_index()
payment['failed_payment_rate']    = payment['failed_payments'] / payment['total_payments']
payment['has_ever_failed_payment'] = (payment['failed_payments'] > 0).astype(int)

# ── Merge all transaction features together ───────────────
txn_features = (last_txn[['customer_id', 'recency_days']]
                .merge(freq,     on='customer_id', how='left')
                .merge(monetary, on='customer_id', how='left')
                .merge(payment[['customer_id', 'failed_payment_rate',
                                 'has_ever_failed_payment']],
                       on='customer_id', how='left'))

print(f"✅ Transaction features built: {txn_features.shape}")
print(f"\n   Feature columns: {txn_features.columns.tolist()}")
print(f"\n   Sample stats:")
print(txn_features.describe().round(2).to_string())

✅ Transaction features built: (5805, 8)

   Feature columns: ['customer_id', 'recency_days', 'transaction_count', 'total_spend', 'avg_spend', 'spend_std', 'failed_payment_rate', 'has_ever_failed_payment']

   Sample stats:
       recency_days  transaction_count  total_spend  avg_spend  spend_std  failed_payment_rate  has_ever_failed_payment
count       5805.00            5805.00      5805.00    5805.00    5805.00              5805.00                  5805.00
mean          43.52              34.86      2420.57      65.34       7.81                 0.03                     0.49
std            9.74              22.18      2110.68      30.00       8.51                 0.06                     0.50
min           28.00               1.00         0.00       0.00       0.00                 0.00                     0.00
25%           34.00              14.00       651.21      38.81       1.78                 0.00                     0.00
50%           45.00              34.00      1661.80      

In [3]:
# ── Cell 3: Session Behavioral Features ───────────────────

ses = sessions[sessions['session_start'] <= CUTOFF].copy()

# Define time windows relative to cutoff
d30 = CUTOFF - pd.Timedelta(days=30)
d60 = CUTOFF - pd.Timedelta(days=60)
d90 = CUTOFF - pd.Timedelta(days=90)

# ── Overall session features ──────────────────────────────
ses_overall = ses.groupby('customer_id').agg(
    total_sessions      = ('session_id', 'count'),
    avg_session_duration = ('duration_minutes', 'mean'),
    avg_pages_viewed    = ('pages_viewed', 'mean'),
    last_session_date   = ('session_start', 'max')
).reset_index()

ses_overall['days_since_last_session'] = (
    CUTOFF - ses_overall['last_session_date']
).dt.days

# ── Sessions in last 30 / 60 / 90 days ───────────────────
ses_30 = (ses[ses['session_start'] >= d30]
          .groupby('customer_id').size()
          .reset_index(name='sessions_last_30d'))

ses_60 = (ses[ses['session_start'] >= d60]
          .groupby('customer_id').size()
          .reset_index(name='sessions_last_60d'))

ses_90 = (ses[ses['session_start'] >= d90]
          .groupby('customer_id').size()
          .reset_index(name='sessions_last_90d'))

# ── Merge all session features ────────────────────────────
ses_features = (ses_overall[['customer_id', 'total_sessions',
                              'avg_session_duration', 'avg_pages_viewed',
                              'days_since_last_session']]
                .merge(ses_30, on='customer_id', how='left')
                .merge(ses_60, on='customer_id', how='left')
                .merge(ses_90, on='customer_id', how='left'))

# Fill NaN (no sessions in that window = 0)
for col in ['sessions_last_30d', 'sessions_last_60d', 'sessions_last_90d']:
    ses_features[col] = ses_features[col].fillna(0).astype(int)

# ── Activity trend: are they becoming more or less active? ─
# Ratio of recent (30d) to older activity — below 1.0 = declining
ses_features['activity_trend_30_90'] = (
    ses_features['sessions_last_30d'] /
    (ses_features['sessions_last_90d'] + 1e-9)   # avoid division by zero
).round(4)

print(f"✅ Session features built: {ses_features.shape}")
print(f"\n   Feature columns: {ses_features.columns.tolist()}")
print(f"\n   Sample stats:")
print(ses_features[['total_sessions', 'avg_session_duration',
                     'days_since_last_session', 'sessions_last_30d',
                     'sessions_last_90d', 'activity_trend_30_90']].describe().round(2).to_string())

✅ Session features built: (4710, 9)

   Feature columns: ['customer_id', 'total_sessions', 'avg_session_duration', 'avg_pages_viewed', 'days_since_last_session', 'sessions_last_30d', 'sessions_last_60d', 'sessions_last_90d', 'activity_trend_30_90']

   Sample stats:
       total_sessions  avg_session_duration  days_since_last_session  sessions_last_30d  sessions_last_90d  activity_trend_30_90
count         4710.00               4710.00                  4710.00            4710.00            4710.00               4710.00
mean           258.77                 30.08                     6.44               5.54              19.00                  0.30
std            187.81                  9.91                     7.78               3.10               6.69                  0.16
min              1.00                  1.00                     0.00               0.00               1.00                  0.00
25%             94.00                 19.67                     1.00               3.00 

In [4]:
# ── Cell 4: Support Ticket Features ───────────────────────

tkt = tickets[tickets['created_date'] <= CUTOFF].copy()

# ── Overall ticket volume and satisfaction ────────────────
tkt_overall = tkt.groupby('customer_id').agg(
    total_tickets       = ('ticket_id', 'count'),
    avg_satisfaction    = ('satisfaction_score', 'mean'),
    min_satisfaction    = ('satisfaction_score', 'min'),   # worst experience ever
    avg_resolution_hrs  = ('resolution_hours', 'mean'),
    last_ticket_date    = ('created_date', 'max')
).reset_index()

tkt_overall['days_since_last_ticket'] = (
    CUTOFF - tkt_overall['last_ticket_date']
).dt.days

# ── Tickets in last 30 / 90 days ──────────────────────────
d30 = CUTOFF - pd.Timedelta(days=30)
d90 = CUTOFF - pd.Timedelta(days=90)

tkt_30 = (tkt[tkt['created_date'] >= d30]
          .groupby('customer_id').size()
          .reset_index(name='tickets_last_30d'))

tkt_90 = (tkt[tkt['created_date'] >= d90]
          .groupby('customer_id').size()
          .reset_index(name='tickets_last_90d'))

# ── Ticket category breakdown ─────────────────────────────
# How many billing vs technical complaints?
tkt_cats = (tkt.groupby(['customer_id', 'category'])
            .size()
            .unstack(fill_value=0)
            .reset_index())

# Rename columns clearly
tkt_cats.columns = ['customer_id'] + [f'tickets_{c.lower()}' for c in tkt_cats.columns[1:]]

# ── Low satisfaction flag: ever rated 1 or 2 ─────────────
bad_sat = (tkt[tkt['satisfaction_score'] <= 2]
           .groupby('customer_id').size()
           .reset_index(name='low_satisfaction_tickets'))

# ── Merge all ticket features ─────────────────────────────
tkt_features = (tkt_overall[['customer_id', 'total_tickets', 'avg_satisfaction',
                               'min_satisfaction', 'avg_resolution_hrs',
                               'days_since_last_ticket']]
                .merge(tkt_30,   on='customer_id', how='left')
                .merge(tkt_90,   on='customer_id', how='left')
                .merge(tkt_cats, on='customer_id', how='left')
                .merge(bad_sat,  on='customer_id', how='left'))

# Fill NaN for window and category columns
fill_zero_cols = ['tickets_last_30d', 'tickets_last_90d', 'low_satisfaction_tickets']
fill_zero_cols += [c for c in tkt_features.columns if c.startswith('tickets_') and c not in fill_zero_cols]
for col in fill_zero_cols:
    if col in tkt_features.columns:
        tkt_features[col] = tkt_features[col].fillna(0).astype(int)

print(f"✅ Ticket features built: {tkt_features.shape}")
print(f"\n   Feature columns: {tkt_features.columns.tolist()}")
print(f"\n   Sample stats:")
print(tkt_features[['total_tickets', 'avg_satisfaction', 'min_satisfaction',
                     'tickets_last_30d', 'tickets_last_90d',
                     'low_satisfaction_tickets']].describe().round(2).to_string())

✅ Ticket features built: (5461, 13)

   Feature columns: ['customer_id', 'total_tickets', 'avg_satisfaction', 'min_satisfaction', 'avg_resolution_hrs', 'days_since_last_ticket', 'tickets_last_30d', 'tickets_last_90d', 'tickets_account', 'tickets_billing', 'tickets_service', 'tickets_technical', 'low_satisfaction_tickets']

   Sample stats:
       total_tickets  avg_satisfaction  min_satisfaction  tickets_last_30d  tickets_last_90d  low_satisfaction_tickets
count        5461.00           5461.00           5461.00           5461.00           5461.00                   5461.00
mean            7.54              3.13              1.76              0.30              0.81                      2.57
std             5.83              0.76              1.02              0.61              1.01                      2.82
min             1.00              1.00              1.00              0.00              0.00                      0.00
25%             3.00              2.60              1.00       

In [5]:
# ── Cell 5: Customer-Level Feature Encoding ───────────────

cust = customers.copy()

# ── 1. Ordinal encoding: Contract type ───────────────────
# EDA showed a clear order: Month-to-month > One year > Two year
# Encode as risk level: higher number = higher churn risk
contract_map = {'Two year': 0, 'One year': 1, 'Month-to-month': 2}
cust['contract_risk'] = cust['Contract'].map(contract_map)

# ── 2. Binary encoding: Yes/No columns ───────────────────
yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'MultipleLines',
               'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
               'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling']

for col in yes_no_cols:
    cust[col + '_bin'] = (cust[col] == 'Yes').astype(int)

# ── 3. Internet service: one-hot (3 categories) ──────────
cust['is_fiber_optic'] = (cust['InternetService'] == 'Fiber optic').astype(int)
cust['is_dsl']         = (cust['InternetService'] == 'DSL').astype(int)
# 'No internet' is the reference category (both = 0)

# ── 4. Payment method: one-hot ────────────────────────────
cust['pays_electronic_check'] = (cust['PaymentMethod'] == 'Electronic check').astype(int)
cust['pays_mailed_check']     = (cust['PaymentMethod'] == 'Mailed check').astype(int)
cust['pays_bank_transfer']    = (cust['PaymentMethod'] == 'Bank transfer (automatic)').astype(int)
# 'Credit card (automatic)' is the reference category

# ── 5. Composite protection score ────────────────────────
# Count how many protective services a customer has
# (EDA showed these reduce churn)
protection_cols = ['OnlineSecurity_bin', 'OnlineBackup_bin',
                   'DeviceProtection_bin', 'TechSupport_bin']
cust['protection_score'] = cust[protection_cols].sum(axis=1)

# ── 6. Select final customer-level features ───────────────
customer_features_cols = [
    'customerID',
    'tenure', 'MonthlyCharges', 'TotalCharges',
    'SeniorCitizen',
    'contract_risk',
    'is_fiber_optic', 'is_dsl',
    'pays_electronic_check', 'pays_mailed_check', 'pays_bank_transfer',
    'Partner_bin', 'Dependents_bin',
    'PhoneService_bin', 'MultipleLines_bin',
    'OnlineSecurity_bin', 'OnlineBackup_bin',
    'DeviceProtection_bin', 'TechSupport_bin',
    'StreamingTV_bin', 'StreamingMovies_bin',
    'PaperlessBilling_bin',
    'protection_score',
    'churn_binary'    # ← this is your target label
]

cust_features = cust[customer_features_cols].copy()

print(f"✅ Customer features built: {cust_features.shape}")
print(f"\n   Feature columns ({len(customer_features_cols)-2} features + ID + label):")
for col in customer_features_cols:
    marker = " ← TARGET" if col == 'churn_binary' else ""
    print(f"     {col}{marker}")

print(f"\n   Missing values: {cust_features.isnull().sum().sum()}")
print(f"   Churn rate:     {cust_features['churn_binary'].mean()*100:.1f}%")

✅ Customer features built: (7043, 24)

   Feature columns (22 features + ID + label):
     customerID
     tenure
     MonthlyCharges
     TotalCharges
     SeniorCitizen
     contract_risk
     is_fiber_optic
     is_dsl
     pays_electronic_check
     pays_mailed_check
     pays_bank_transfer
     Partner_bin
     Dependents_bin
     PhoneService_bin
     MultipleLines_bin
     OnlineSecurity_bin
     OnlineBackup_bin
     DeviceProtection_bin
     TechSupport_bin
     StreamingTV_bin
     StreamingMovies_bin
     PaperlessBilling_bin
     protection_score
     churn_binary ← TARGET

   Missing values: 0
   Churn rate:     26.5%


In [6]:
# ── Cell 6: Merge All Features into Master Table ──────────

# Start from customer features (all 7,043 rows)
# Left join everything else — customers with no records get filled with 0s
master = cust_features.copy()

# ── Merge transaction features ────────────────────────────
master = master.merge(txn_features, left_on='customerID',
                      right_on='customer_id', how='left')
master.drop(columns=['customer_id'], inplace=True)

# ── Merge session features ────────────────────────────────
master = master.merge(ses_features, left_on='customerID',
                      right_on='customer_id', how='left')
master.drop(columns=['customer_id'], inplace=True)

# ── Merge ticket features ─────────────────────────────────
master = master.merge(tkt_features, left_on='customerID',
                      right_on='customer_id', how='left')
master.drop(columns=['customer_id'], inplace=True)

# ── Fill NaN for customers with no records in a table ─────
# Transaction defaults
txn_fill = {
    'recency_days': 999,          # 999 = never transacted (very stale)
    'transaction_count': 0,
    'total_spend': 0,
    'avg_spend': 0,
    'spend_std': 0,
    'failed_payment_rate': 0,
    'has_ever_failed_payment': 0
}

# Session defaults
ses_fill = {
    'total_sessions': 0,
    'avg_session_duration': 0,
    'avg_pages_viewed': 0,
    'days_since_last_session': 999,   # 999 = never logged in
    'sessions_last_30d': 0,
    'sessions_last_60d': 0,
    'sessions_last_90d': 0,
    'activity_trend_30_90': 0
}

# Ticket defaults
tkt_fill = {
    'total_tickets': 0,
    'avg_satisfaction': 3.0,          # neutral score for no tickets
    'min_satisfaction': 3.0,
    'avg_resolution_hrs': 0,
    'days_since_last_ticket': 999,
    'tickets_last_30d': 0,
    'tickets_last_90d': 0,
    'tickets_account': 0,
    'tickets_billing': 0,
    'tickets_service': 0,
    'tickets_technical': 0,
    'low_satisfaction_tickets': 0
}

master.fillna({**txn_fill, **ses_fill, **tkt_fill}, inplace=True)

# ── Final check ───────────────────────────────────────────
total_features = master.shape[1] - 2  # subtract customerID and churn_binary
missing        = master.isnull().sum().sum()
churn_rate     = master['churn_binary'].mean() * 100

print(f"✅ Master feature table built!")
print(f"\n   Shape:          {master.shape}")
print(f"   Features:       {total_features}")
print(f"   Missing values: {missing}")
print(f"   Churn rate:     {churn_rate:.1f}%")

# ── Save to disk ──────────────────────────────────────────
output_path = r'/Users/siddharthm/Desktop/churn-prediction/data/processed/features.csv'
master.to_csv(output_path, index=False)

print(f"\n   ✅ Saved to: data/processed/features.csv")

# ── Feature summary ───────────────────────────────────────
print(f"\n{'='*55}")
print(f"  FEATURE GROUPS SUMMARY")
print(f"{'='*55}")
print(f"  Customer base features:    22  (tenure, contract, services...)")
print(f"  Transaction / RFM:          7  (recency, frequency, monetary...)")
print(f"  Session behavioral:         8  (activity windows, trend...)")
print(f"  Support ticket:            12  (satisfaction, volume, category...)")
print(f"  {'─'*40}")
print(f"  Total features:            {total_features}")
print(f"\n  Customers in dataset:   {len(master):,}")
print(f"  Churned:                {master['churn_binary'].sum():,}  ({churn_rate:.1f}%)")
print(f"  Retained:               {(master['churn_binary']==0).sum():,}  ({100-churn_rate:.1f}%)")

✅ Master feature table built!

   Shape:          (7043, 51)
   Features:       49
   Missing values: 0
   Churn rate:     26.5%

   ✅ Saved to: data/processed/features.csv

  FEATURE GROUPS SUMMARY
  Customer base features:    22  (tenure, contract, services...)
  Transaction / RFM:          7  (recency, frequency, monetary...)
  Session behavioral:         8  (activity windows, trend...)
  Support ticket:            12  (satisfaction, volume, category...)
  ────────────────────────────────────────
  Total features:            49

  Customers in dataset:   7,043
  Churned:                1,869  (26.5%)
  Retained:               5,174  (73.5%)
